# 📘 Risk Analysis — Winter Semester 2025/2026
## Tutorial 9 ( In class coding activity): Uncertainty Propagation

**Supervisor:** Dr. Daniel Straub  
**Conducted by:** Milad Cheraghzade  
**Date:** December 18th, 2025  
**Time:** 15:00–16:00  
**Location:** TUM, Room N 1070  

---


In [1]:
# ============================================================
# Colab Setup Cell (run once)
# - Installs/ensures required libraries are available
# - Note: In Google Colab, numpy and matplotlib are usually pre-installed.
# ============================================================

# If you're on Colab, you can run this cell safely even if packages exist.
!pip -q install numpy matplotlib scipy

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import uniform

print('✅ Setup complete: numpy, matplotlib, scipy imported successfully.')

# ------------------------------------------------------------
# Tip for students:
# You can save your own version of this notebook:
#   File  -> Save a copy in Drive
# or download it:
#   File  -> Download -> .ipynb
# Then you can edit and keep your own version of the code.
# ------------------------------------------------------------


✅ Setup complete: numpy, matplotlib, scipy imported successfully.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import uniform

# ============================================================
# Problem 5.1 — Concentration of particles
#
#   X ~ Uniform[a, b] with a=10, b=50
#   C = g_C(X) = c0 * exp(-X),  c0 = 1e-2
#
# Goal: reproduce the analytical derivation in code:
#   Step 1) Inversion:      x = h(c) = -ln(c/c0)
#   Step 2) CDF:            F_C(c) = P(C <= c) = 1 - F_X(h(c))
#   Step 3) PDF (Jacobian): f_C(c) = f_X(h(c)) * |dh/dc|
# ============================================================

# -----------------------------
# Given constants
# -----------------------------
c0 = 1e-2
a, b = 10.0, 50.0

# -----------------------------
# X distribution (Uniform)
# -----------------------------
# f_X(x) = 1/(b-a) on [a,b]
def pdf_X(x):
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(x)
    mask = (x >= a) & (x <= b)
    out[mask] = 1.0 / (b - a)
    return out

def cdf_X(x):
    # Uniform CDF on [a,b]
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(x)
    out[x < a] = 0.0
    out[x > b] = 1.0
    mask = (x >= a) & (x <= b)
    out[mask] = (x[mask] - a) / (b - a)
    return out


# ============================================================
# Step 1 — Inversion: c = c0 * exp(-x)  =>  x = h(c) = -ln(c/c0)
# ============================================================
def h(c):
    # inverse mapping x = h(c)
    c = np.asarray(c, dtype=float)
    return -np.log(c / c0)

# Support of C from X in [a,b]:
#   c_min = c0 * exp(-b), c_max = c0 * exp(-a)
c_min = c0 * np.exp(-b)
c_max = c0 * np.exp(-a)

print('Step 1: Inversion')
print('  h(c) = -ln(c/c0)')
print('  Support of C: [c_min, c_max] =', (c_min, c_max))


# ============================================================
# Task 1 (Student exercise): Implement the CDF of C
#
# Question:
#   Implement the CDF
#       F_C(c) = P(C <= c)
#   using the analytical result (g is strictly decreasing):
#       F_C(c) = 1 - F_X(h(c)),
#   where h(c) = -ln(c/c0).
#
# Hint:
#   You already have the CDF of X implemented as:
#       cdf_X(x)
#   Use it directly:
#       F_C(c) = 1 - cdf_X(h(c))   for c in [c_min, c_max].
#
# Requirements:
#   - Return 0 for c < c_min
#   - Return 1 for c > c_max
#   - Use 1 - cdf_X(h(c)) on the support
# ============================================================

# --- Your code starts here Task 1 ---


# --- Your code ends here ---

# ============================================================
# Step 3 — PDF via Jacobian:
#   f_C(c) = f_X(h(c)) * |dh/dc|
#
# Derivative:
#   d/dc ln(c/c0) = 1/c
#   dh/dc = d/dc[-ln(c/c0)] = -1/c
#   |dh/dc| = 1/c
# ============================================================
def dh_dc(c):
    c = np.asarray(c, dtype=float)
    return -1.0 / c  # dh/dc

def pdf_C(c):
    c = np.asarray(c, dtype=float)
    out = np.zeros_like(c)

    mask = (c >= c_min) & (c <= c_max)
    out[mask] = pdf_X(h(c[mask])) * np.abs(dh_dc(c[mask]))  # = (1/(b-a))*(1/c)
    return out

print('\nStep 3: PDF')
print('  dh/dc = -1/c, |dh/dc| = 1/c')
print('  f_C(c) = f_X(h(c)) * |dh/dc| = 1/((b-a)*c) on [c_min, c_max]')


# ============================================================
# Task 2 (Student exercise): Check that f_C(c) is a proper PDF
#
# Question:
#   Verify numerically that the analytical PDF integrates to 1:
#       ∫_{c_min}^{c_max} f_C(c) dc = 1.
#
# Hint:
#   Use a fine grid on [c_min, c_max] and approximate the integral.
# ============================================================

# --- Your code starts here ---


# --- Your code ends here ---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Problem 5.2 — Rockfall energy
#
# Energy:
#   E(M, V) = (M * V^2) / 2
#
# Given:
#   mu_M = 100 kg,  sigma_M = 100 kg
#   mu_V = 20 m/s,  sigma_V = 10  m/s
#   Corr(M,V) = rho   (cases: rho = 0 and rho = -0.5)
#
# Tasks:
#   (A) Second-order (Taylor) approximation of mean:
#       mu_E ≈ E(mu) + 1/2 * tr(B * Sigma)
#
#   (B) First-order (linear) approximation of std:
#       sigma_E ≈ sqrt( a^T * Sigma * a )
#
# where:
#   a = grad E evaluated at mu
#   B = Hessian of E evaluated at mu
# ============================================================

# -----------------------------
# Given statistics
# -----------------------------
mu_M, sigma_M = 100.0, 100.0
mu_V, sigma_V = 20.0, 10.0

mu_X = np.array([mu_M, mu_V])

# -----------------------------
# Model function E(M, V)
# -----------------------------
def E(M, V):
    return 0.5 * M * V**2

# -----------------------------
# Gradient and Hessian (symbolic results)
#   dE/dM = V^2 / 2
#   dE/dV = M V
#
#   d2E/dM2     = 0
#   d2E/dV2     = M
#   d2E/dMdV    = V   (and symmetric)
# -----------------------------
def grad_E_at_mu(mu_M, mu_V):
    a1 = 0.5 * mu_V**2
    a2 = mu_M * mu_V
    return np.array([a1, a2])

def hessian_E_at_mu(mu_M, mu_V):
    b11 = 0.0
    b12 = mu_V
    b21 = mu_V
    b22 = mu_M
    return np.array([[b11, b12],
                     [b21, b22]])

a = grad_E_at_mu(mu_M, mu_V)
B = hessian_E_at_mu(mu_M, mu_V)

# -----------------------------
# Utility: build covariance matrix Sigma for a given rho
# -----------------------------
def Sigma_from_rho(rho, sigma_M, sigma_V):
    cov_MV = rho * sigma_M * sigma_V
    return np.array([[sigma_M**2, cov_MV],
                     [cov_MV,      sigma_V**2]])

# ============================================================
# TASK 3 (Student task)
#
# Implement the Taylor approximations for the rockfall energy E.
#
# For a given correlation coefficient rho:
#   1) Construct the covariance matrix Sigma_XX.
#   2) Evaluate the energy E at the mean values (mu_M, mu_V).
#   3) Compute the second-order approximation of the mean:
#        mu_E ≈ E(mu) + 0.5 * tr(B @ Sigma)
#   4) Compute the first-order approximation of the standard deviation:
#        sigma_E ≈ sqrt(a^T @ Sigma @ a)
#
# Return E(mu), Sigma, mu_E, and sigma_E.
# ============================================================


# --- Your code starts here ---


# --- Your code ends here ---





# -----------------------------
# Run the two required cases
# -----------------------------
for rho in [0.0, -0.5]:
    E_mu, Sigma, mu_E, sigma_E = taylor_approximations(rho)

    print('\n' + '='*60)
    print(f'Case rho = {rho}')
    print('='*60)

    print('E(mu) =', E_mu)

    print('\nCovariance matrix Sigma_XX =')
    print(Sigma)

    print('\nGradient a = grad E |_{mu} =')
    print(a)

    print('\nHessian B = Hessian E |_{mu} =')
    print(B)

    print('\nSecond-order mean approximation:')
    print('mu_E ≈ E(mu) + 0.5 * tr(B Sigma) =', mu_E)

    print('\nFirst-order standard deviation approximation:')
    print('sigma_E ≈ sqrt(a^T Sigma a) =', sigma_E)

# -----------------------------
# Monte Carlo Simulation for Joint Distribution and E Distribution
# -----------------------------
np.random.seed(0)  # For reproducibility

for rho in [0.0, -0.5]:
    # Get the covariance matrix for this rho
    Sigma = Sigma_from_rho(rho, sigma_M, sigma_V)

    # Generate joint samples of (M,V)
    mean = [mu_M, mu_V]
    samples = np.random.multivariate_normal(mean, Sigma, size=10000)

    M_samples = samples[:, 0]
    V_samples = samples[:, 1]

    # Compute E for each (M,V) sample
    E_samples = E(M_samples, V_samples)

    # Plot the joint distribution of (M,V)
    plt.figure(figsize=(7, 6))
    plt.hexbin(M_samples, V_samples, gridsize=50, cmap='Blues', mincnt=1)
    plt.colorbar(label='Count in bin')
    plt.xlabel('Mass M (kg)')
    plt.ylabel('Velocity V (m/s)')
    plt.title(f'Joint distribution of (M,V) for rho={rho}')
    plt.tight_layout()
    plt.show()

    # Plot the distribution of E
    plt.figure(figsize=(7, 5))
    plt.hist(E_samples, bins=50, density=True, alpha=0.7, color='green', edgecolor='black')
    plt.xlabel('Energy E (J)')
    plt.ylabel
